In [1]:
import pandas as pd
import numpy as np
import json
import seaborn as sns
import matplotlib.pyplot as plt

# Warning off
pd.options.mode.chained_assignment = None

import warnings
warnings.filterwarnings("ignore")

In [7]:
def process_filtered_df(results_df,df_def,defense,narcissus,header=None,header_idx=0):
    if narcissus:
        avg_poison_success = df_def['P1 Acc'].mean() * 100
        std_poison_success = df_def['P1 Acc'].std() * 100
        avg_nat_acc = df_def['End Acc'].mean()
        std_nat_acc = df_def['End Acc'].std()
        max_poison_success = df_def['P1 Acc'].max() * 100


        if header is None:
            results_df.loc[defense, 'Avg Poison Success'] = f'{avg_poison_success:.2f}\u00B1{std_poison_success:.1f}'
            results_df.loc[defense,'Avg Nat Acc'] = f'{avg_nat_acc:.2f}\u00B1{std_nat_acc:.2f}'
            results_df.loc[defense,  'Max Poison Success'] = f'{max_poison_success:.2f}'

        else:
            results_df.loc[defense, (header_idx, 'Avg Poison Success')] = f'{avg_poison_success:.2%}\u00B1{std_poison_success:.1f}'
            results_df.loc[defense, (header_idx, 'Avg Nat Acc')] = f'{avg_nat_acc:.2f}\u00B1{std_nat_acc:.1f}'
            results_df.loc[defense, (header_idx, 'Max Poison Success')] = f'{max_poison_success:.2f}'

    else:

        success_rate = (df_def["Success"] == True).sum() / len(df_def)
        avg_nat_acc = df_def["End Acc"].mean()
        std_nat_acc = df_def["End Acc"].std()

        if header is None:
            results_df.loc[defense, 'Poison Success'] = f'{success_rate:.2%}'
            results_df.loc[defense, 'Natural Accuracy'] = f'{avg_nat_acc:.2f}%\u00B1{std_nat_acc:.1f}'

        else:
            results_df.loc[defense, (header_idx, 'Poison Success')] = f'{success_rate:.2%}'
            results_df.loc[defense, (header_idx, 'Natural Accuracy')] = f'{avg_nat_acc:.2f}%\u00B1{std_nat_acc:.1f}'

def export_results(df, narcissus = False, header=None, filters = {}):

    # Iterate through the filters
    for key, value in filters.items():
        df = df[df[key] == value]

    # Initialize an empty DataFrame for results
    if narcissus:
        if header is None:
            columns = ['Avg Poison Success', 'Avg Nat Acc','Max Poison Success']
        else:
            columns = pd.MultiIndex.from_product([np.sort(df[header].unique()), ['Avg Poison Success', 'Avg Nat Acc','Max Poison Success']],
                                     names=[header, 'Metric'])

    else:
        if header is None:
            columns = ['Poison Success', 'Natural Accuracy']
        else:
            columns = pd.MultiIndex.from_product([df[header].unique(), ['Poison Success', 'Natural Accuracy']],
                                                names=[header, 'Metric'])

    # Define the defenses
    defenses = ['None', 'EBM','Diff','EBM_Diff']                                         
    
    # Populate the DataFrame
    results_df = pd.DataFrame(columns=columns)

    if header is None: header_idxs = [0]
    else: header_idxs = df[header].unique()

    for header_idx in header_idxs:

        for defense in defenses:
            if header is None:
                if defense == 'Diff':
                    for steps in np.sort(df['Steps'].unique()):
                        df_def = df[(df['Defense'] == defense) & (df['Steps'] == steps)]
                        process_filtered_df(results_df,df_def,defense+f'-{steps}',narcissus,header,header_idx)
                        print(f'Added {defense}-{steps} to results_df with num rows {len(df_def)}')
                elif defense == 'EBM_Diff':
                    for steps in np.sort(df['Steps'].unique()):
                        df_def = df[(df['Defense'] == defense) & (df['Steps'] == steps)]
                        process_filtered_df(results_df,df_def,defense+f'-{steps}',narcissus,header,header_idx)
                        print(f'Added {defense}-{steps} to results_df with num rows {len(df_def)}')
                else:
                    df_def = df[(df['Defense'] == defense)]
                    process_filtered_df(results_df,df_def,defense,narcissus,header,header_idx)
                    print(f'Added {defense} to results_df with num rows {len(df_def)}')

            else:
                df_def = df[(df['Defense'] == 'None') & (df[header] == header_idx)]
                process_filtered_df(results_df,df_def,defense,narcissus,header,header_idx)

    return results_df

In [3]:
df = pd.read_csv('/home/sunaybhat/results_EBM_Defense/From_Scratch/Narcissus/Results.csv')

df['Defense'] = df['Defense'].fillna('None')
df['Steps'] = df['subset_size'] = df['Args'].apply(lambda x: json.loads(x)['diff_steps'] if 'diff_steps' in x else np.nan)
df['Steps'] = df['Steps'].fillna(125)


columns = ['Avg Poison Success', 'Avg Nat Acc','Max Poison Success']
results_df = pd.DataFrame(columns=columns)
for steps in np.sort(df['Steps'].unique()):
    df_def = df[(df['Defense'] == defense) & (df['Steps'] == steps)]
    process_filtered_df(results_df,df_def,defense+f'-{steps}',True)
    print(f'Added {defense}-{steps} to results_df with num rows {len(df_def)}')
df_results = export_results(df,narcissus=True)

df_results

Added None to results_df with num rows 10
Added EBM to results_df with num rows 10
Added Diff-10.0 to results_df with num rows 10
Added Diff-25.0 to results_df with num rows 10
Added Diff-50.0 to results_df with num rows 10
Added Diff-100.0 to results_df with num rows 10
Added Diff-125.0 to results_df with num rows 10
Added Diff-150.0 to results_df with num rows 10
Added Diff-175.0 to results_df with num rows 10
Added Diff-200.0 to results_df with num rows 8
Added EBM_Diff-10.0 to results_df with num rows 10
Added EBM_Diff-25.0 to results_df with num rows 10
Added EBM_Diff-50.0 to results_df with num rows 10
Added EBM_Diff-100.0 to results_df with num rows 10
Added EBM_Diff-125.0 to results_df with num rows 10
Added EBM_Diff-150.0 to results_df with num rows 10
Added EBM_Diff-175.0 to results_df with num rows 10
Added EBM_Diff-200.0 to results_df with num rows 0


,Avg Poison Success,Avg Nat Acc,Max Poison Success
None,43.95±33.6,94.89±0.23,93.59
EBM,1.34±0.6,92.73±0.22,2.29
Diff-10.0,42.63±47.5,94.42±0.13,99.76
Diff-25.0,42.08±48.6,94.04±0.16,99.99
Diff-50.0,22.66±39.8,93.67±0.13,99.86
Diff-100.0,2.36±2.4,93.64±0.13,8.92
Diff-125.0,2.00±2.0,93.50±0.16,7.49
Diff-150.0,1.96±1.9,93.36±0.23,7.16
Diff-175.0,1.72±1.4,93.24±0.26,5.42
Diff-200.0,1.52±0.9,93.27±0.27,3.69


In [12]:
df = pd.read_csv('/Users/sunaybhat/Desktop/Results_yuan.csv')

df['Defense'] = df['Defense'].fillna('None')
df['Steps'] = df['Args'].apply(lambda x: json.loads(x)['num_t_steps'] if 'num_t_steps' in x else np.nan)
df['EBM'] = df['Args'].apply(lambda x: json.loads(x)['ebm_type'] if 'ebm_type' in x else np.nan)

columns = ['Avg Poison Success', 'Avg Nat Acc','Max Poison Success']
results_df = pd.DataFrame(columns=columns)
for ebm in np.sort(df['EBM'].unique()):
    for steps in np.sort(df['Steps'].unique()):
        df_def = df[(df['EBM'] == ebm) & (df['Steps'] == steps)]
        process_filtered_df(results_df,df_def,f'{ebm}-{steps}',True)
        print(f'Added {ebm}-{steps} to results_df with num rows {len(df_def)}')

results_df

Added EBMSNGAN32-1000 to results_df with num rows 30
Added EBM_Small-1000 to results_df with num rows 10


,Avg Poison Success,Avg Nat Acc,Max Poison Success
EBMSNGAN32-1000,1.29±0.7,92.91±0.16,2.73
EBM_Small-1000,1.29±0.5,92.94±0.23,2.35


In [9]:
df

,Defense,Target Index,End Acc,P1 Acc,T1 Acc,Exp Name,Calc Time,Train Time,P2 Acc,T2 Acc,P3 Acc,T3 Acc,Args,Logs,Steps,subset_size,EBM
0,EBM_Diff,7,92.85,0.007724,0.917760,narcissus8-sebmdiff150,2024_02_22_19_26,5505.106470,0.011906,0.879005,0.018178,0.780942,"{""remote_user"": ""alice"", ""config_file"": ""./Con...","{""train_loss"": [2.1766599947229373, 1.53499831...",NaN,NaN,EBM_Small
1,EBM_Diff,3,92.80,0.023548,0.931272,narcissus8-sebmdiff150,2024_02_22_19_26,5381.913307,0.048746,0.884441,0.094586,0.792738,"{""remote_user"": ""alice"", ""config_file"": ""./Con...","{""train_loss"": [1.995302487517257, 1.471154875...",NaN,NaN,EBM_Small
2,EBM_Diff,6,92.85,0.018178,0.903059,narcissus8-sebmdiff150,2024_02_22_19_26,5383.489468,0.144564,0.731272,0.336620,0.460101,"{""remote_user"": ""alice"", ""config_file"": ""./Con...","{""train_loss"": [2.113432114691381, 1.619658673...",NaN,NaN,EBM_Small
3,EBM_Diff,5,92.94,0.014877,0.930062,narcissus8-sebmdiff150,2024_02_22_19_26,5509.982319,0.024890,0.895004,0.037676,0.818310,"{""remote_user"": ""alice"", ""config_file"": ""./Con...","{""train_loss"": [1.9586646925762792, 1.41083328...",NaN,NaN,EBM_Small
4,EBM_Diff,4,93.06,0.013996,0.919586,narcissus8-sebmdiff150,2024_02_22_19_26,5512.270754,0.065405,0.838886,0.150022,0.679643,"{""remote_user"": ""alice"", ""config_file"": ""./Con...","{""train_loss"": [1.9061439101348447, 1.32399762...",NaN,NaN,EBM_Small
5,EBM_Diff,1,93.04,0.007923,0.911334,narcissus8-sebmdiff150,2024_02_22_19_26,5474.408305,0.023460,0.827245,0.074428,0.637148,"{""remote_user"": ""alice"", ""config_file"": ""./Con...","{""train_loss"": [1.9215604103434727, 1.41344956...",NaN,NaN,EBM_Small
6,EBM_Diff,2,92.49,0.014085,0.921501,narcissus8-sebmdiff150,2024_02_22_19_26,5509.285694,0.043398,0.866285,0.100836,0.733319,"{""remote_user"": ""alice"", ""config_file"": ""./Con...","{""train_loss"": [2.065028467751525, 1.483485010...",NaN,NaN,EBM_Small
7,EBM_Diff,0,93.39,0.013226,0.918090,narcissus8-sebmdiff150,2024_02_22_19_26,5373.882038,0.028873,0.848151,0.077883,0.705942,"{""remote_user"": ""alice"", ""config_file"": ""./Con...","{""train_loss"": [2.0185581884725625, 1.46464156...",NaN,NaN,EBM_Small
8,EBM_Diff,9,92.97,0.008473,0.914283,narcissus8-sebmdiff150,2024_02_23_05_24,4927.839193,0.040273,0.801100,0.079467,0.592033,"{""remote_user"": ""alice"", ""config_file"": ""./Con...","{""train_loss"": [1.875944793071893, 1.356907082...",NaN,NaN,EBM_Small
9,EBM_Diff,8,93.03,0.006712,0.915933,narcissus8-sebmdiff150,2024_02_23_05_24,4808.195618,0.016417,0.850352,0.038974,0.713930,"{""remote_user"": ""alice"", ""config_file"": ""./Con...","{""train_loss"": [1.8572408915175806, 1.32202747...",NaN,NaN,EBM_Small
